# 🛰️ SatQuery AI — Kaggle GPU Backend Server

This notebook runs the **full production backend with real deep-learning models** on **Kaggle's free 16GB GPU** (NVIDIA Tesla T4/P100).

> **Prerequisite:** In Kaggle's right-hand sidebar, verify that:
> 1. **Accelerator** is set to **GPU T4 x2** (or GPU P100)
> 2. **Internet** toggle is set to **ON**

---

### Step 1: Verify NVIDIA GPU & Clone Project Codebase

In [ ]:
!nvidia-smi
%cd /kaggle/working
!rm -rf novex
!git clone https://github.com/sanskar620/novex.git
%cd /kaggle/working/novex

### Step 2: Install Geospatial & ML Dependencies
This cell installs GDAL, Cloudflare Tunnel, PyTorch, and Transformers compatible with any Python version.

In [ ]:
# System GIS packages & Cloudflare Tunnel binary
!apt-get update -qq && apt-get install -y -qq gdal-bin libgdal-dev libgeos-dev libproj-dev > /dev/null
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Install backend dependencies
!pip install -q --no-deps -e backend/
!pip install -q fastapi uvicorn pydantic pydantic-settings sqlalchemy alembic redis python-multipart numpy pillow matplotlib rasterio shapely geopandas pyproj reportlab httpx
!pip install -q torch torchvision transformers accelerate timm

### Step 3: Model Selection & Pre-Downloading
We load two real specialist models:
1. **Vision-Language Model:** `OpenGVLab/InternVL2-1B` (Adapted for remote sensing, VQA, scene captioning, spatial bounding boxes)
2. **Task Planning Agent:** `Qwen/Qwen2.5-1.5B-Instruct` (Interprets user queries and orchestrates spatial pipelines)

In [ ]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
import torch

INTERNVL_MODEL = "OpenGVLab/InternVL2-1B"
QWEN_AGENT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"📥 [1/2] Pre-downloading Vision-Language Model: {INTERNVL_MODEL} ...")
AutoTokenizer.from_pretrained(INTERNVL_MODEL, trust_remote_code=True)
AutoModel.from_pretrained(INTERNVL_MODEL, trust_remote_code=True, torch_dtype=torch.float16)

print(f"📥 [2/2] Pre-downloading Agent Planner Model: {QWEN_AGENT_MODEL} ...")
AutoTokenizer.from_pretrained(QWEN_AGENT_MODEL)
AutoModelForCausalLM.from_pretrained(QWEN_AGENT_MODEL, torch_dtype=torch.float16)

print("\n✅ Both real neural models downloaded and cached in GPU memory successfully!")

### Step 4: Configure Environment Variables (Real GPU Mode Active)

In [ ]:
import os

os.environ["DEMO_MODE"] = "false"
os.environ["ENVIRONMENT"] = "production"
os.environ["DEVICE"] = "cuda"
os.environ["INTERNVL_MODEL_PATH"] = INTERNVL_MODEL
os.environ["AGENT_MODEL_NAME"] = "Qwen2.5-1.5B"
os.environ["AGENT_MODEL_PATH"] = QWEN_AGENT_MODEL
os.environ["CORS_ALLOWED_ORIGINS"] = "https://novex-teal.vercel.app,http://localhost:3000"

print("✅ Configuration Active:")
print(f" • DEMO_MODE: {os.environ['DEMO_MODE']} (Real Neural Inference Enabled)")
print(f" • DEVICE: {os.environ['DEVICE']} (CUDA Hardware Acceleration)")
print(f" • Vision Model: {INTERNVL_MODEL}")
print(f" • Agent Model: {QWEN_AGENT_MODEL}")

### Step 5: Start Public Cloudflare Tunnel & Launch Backend Server
Run this cell. It will print a public **`https://xxxx.trycloudflare.com`** URL.
Copy that URL and add it to your Vercel project!

In [ ]:
import subprocess
import threading
import time
import re

def monitor_tunnel():
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*70)
            print("🚀 YOUR LIVE KAGGLE GPU BACKEND IS ONLINE!")
            print(f"👉 Public Backend URL: {url}")
            print(f"👉 Set this in Vercel (NEXT_PUBLIC_API_BASE_URL): {url}/api/v1")
            print("="*70 + "\n")

tunnel_thread = threading.Thread(target=monitor_tunnel, daemon=True)
tunnel_thread.start()
time.sleep(3)

# Start FastAPI Uvicorn Server
%cd /kaggle/working/novex/backend
!python -m uvicorn app.main:app --host 0.0.0.0 --port 8000